In [1]:
import numpy as np
import time
from numba import njit, cuda

In [2]:
@njit
def cpu_sum(arr):
    s = 0.0
    for i in range(len(arr)):
        s += arr[i]
    return s


def run_cpu(arr):
    start = time.time()
    result = cpu_sum(arr)
    end = time.time()
    
    print("CPU Result:", result)
    print("CPU Time:", end - start)

In [ ]:
@cuda.jit
def gpu_sum_kernel(arr, result):
    i = cuda.grid(1)
    if i < arr.size:
        cuda.atomic.add(result, 0, arr[i])


def run_gpu(arr):
    d_arr = cuda.to_device(arr)
    d_result = cuda.to_device(np.array([0.0]))

    threads_per_block = 256
    blocks_per_grid = (arr.size + threads_per_block - 1) // threads_per_block

    start = time.time()

    gpu_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_result)
    cuda.synchronize()

    end = time.time()

    result = d_result.copy_to_host()

    print("GPU Result:", result[0])
    print("GPU Time:", end - start)

In [4]:
if __name__ == "__main__":
    arr = np.random.rand(10**7).astype(np.float32)

    print("Running CPU Version...")
    run_cpu(arr)

    print("\nRunning GPU Version...")
    run_gpu(arr)

Running CPU Version...
CPU Result: 5000753.183208851
CPU Time: 1.7902379035949707

Running GPU Version...
GPU Result: 5000753.183208877
GPU Time: 1.0876476764678955
